# 01 — BOSSbase / BOSSbase-derived dataset preparation

**v0.1.5:** image collections are inspected and streamed directly from the ZIP whenever possible. This avoids traversing/extracting 30,000 JPEG files on a Docker Desktop Windows bind mount.

For the current derivative bundle, `AUTO_HIGHEST_QF` selects the highest detected QF collection as the primary dataset. The selected 10,000 decoded grayscale arrays are saved as lossless PNG and verified pixel-for-pixel after reload. The original JPEG source is still recorded as lossy in metadata.


> **v0.2.0 note:** this notebook is still the only place that defines the 6000/2000/2000 source split. Subsequent notebooks must reuse `source_id` and `split` without reshuffling.

In [1]:
from pathlib import Path
import pandas as pd
from rdhlab.dataset import (
    find_bossbase_archive, discover_image_collections, prepare_bossbase, result_as_dict
)

raw_dir = Path('/workspace/data/raw')
archive = find_bossbase_archive(raw_dir)
print('Detected archive:', archive)
print('Archive size: %.1f MiB' % (archive.stat().st_size / 1024**2))
print('The preparation step will inspect candidate 10,000-image collections after extraction.')


Detected archive: /workspace/data/raw/bossbase.zip
Archive size: 411.7 MiB
The preparation step will inspect candidate 10,000-image collections after extraction.


In [2]:
# Explicit scientific choice for derivative JPEG bundles.
# Use e.g. 'grayscale/QF75' to select a specific variant instead.
SOURCE_SUBDIR = 'AUTO_HIGHEST_QF'

result = prepare_bossbase(
    raw_dir=raw_dir,
    processed_dir='/workspace/data/processed/BOSSbase_primary',
    archive_path=archive,
    expected_count=10_000,
    expected_shape=None,  # auto-detect native dimensions; do not resize
    seed=20260916,
    source_subdir=SOURCE_SUBDIR,
    allow_lossy_source=True,
)

info = result_as_dict(result)
info


Using direct ZIP streaming for 10000 images from grayscale/QF95. The previously extracted raw directory is ignored.
Auto-detected source image shape: 256x256 pixels.
Prepared   500/10000 images...
Prepared  1000/10000 images...
Prepared  1500/10000 images...
Prepared  2000/10000 images...
Prepared  2500/10000 images...
Prepared  3000/10000 images...
Prepared  3500/10000 images...
Prepared  4000/10000 images...
Prepared  4500/10000 images...
Prepared  5000/10000 images...
Prepared  5500/10000 images...
Prepared  6000/10000 images...
Prepared  6500/10000 images...
Prepared  7000/10000 images...
Prepared  7500/10000 images...
Prepared  8000/10000 images...
Prepared  8500/10000 images...
Prepared  9000/10000 images...
Prepared  9500/10000 images...
Prepared 10000/10000 images...


{'archive_path': '/workspace/data/raw/bossbase.zip',
 'archive_sha256': '0def1913155a1bd1711a584080a2d286b7e41c3223d6710d68468945f22a5d37',
 'extracted_dir': '/workspace/data/raw/BOSSbase_1.01_extracted',
 'source_dir': 'zip:///workspace/data/raw/bossbase.zip!/grayscale/QF95',
 'source_label': 'BOSSbase-derived JPEG (grayscale/QF95, QF95)',
 'source_is_lossy': True,
 'prepared_dir': '/workspace/data/processed/BOSSbase_primary/png',
 'manifest_path': '/workspace/data/processed/BOSSbase_primary/dataset_manifest.csv',
 'metadata_path': '/workspace/data/processed/BOSSbase_primary/dataset_metadata.json',
 'image_count': 10000,
 'train_count': 6000,
 'validation_count': 2000,
 'test_count': 2000,
 'converted_count': 10000,
 'reused_count': 0}

In [3]:
manifest = pd.read_csv(result.manifest_path)
counts = manifest['split'].value_counts().to_dict()
assert len(manifest) == 10_000
assert counts == {'train': 6000, 'validation': 2000, 'test': 2000}
assert manifest['width'].nunique() == 1
assert manifest['height'].nunique() == 1
print('Native image size:', int(manifest['width'].iloc[0]), 'x', int(manifest['height'].iloc[0]))
assert (manifest['dtype'] == 'uint8').all()
assert manifest['source_id'].nunique() == 10_000
print('Dataset integrity checks: PASS')
print('Source label:', result.source_label)
print('Source collection:', result.source_dir)
print('Source access mode: zip-direct (preferred)')
print('Source is lossy JPEG:', result.source_is_lossy)
print('Split counts:', counts)
print('Manifest:', result.manifest_path)
print('Metadata:', result.metadata_path)
if result.source_is_lossy:
    print('IMPORTANT: cite/report this as a BOSSbase-derived JPEG dataset, not canonical lossless BOSSbase 1.01.')
manifest.head()


Native image size: 256 x 256
Dataset integrity checks: PASS
Source label: BOSSbase-derived JPEG (grayscale/QF95, QF95)
Source collection: zip:///workspace/data/raw/bossbase.zip!/grayscale/QF95
Source access mode: zip-direct (preferred)
Source is lossy JPEG: True
Split counts: {'train': 6000, 'test': 2000, 'validation': 2000}
Manifest: /workspace/data/processed/BOSSbase_primary/dataset_manifest.csv
Metadata: /workspace/data/processed/BOSSbase_primary/dataset_metadata.json
IMPORTANT: cite/report this as a BOSSbase-derived JPEG dataset, not canonical lossless BOSSbase 1.01.


,source_id,split,source_path,source_format,source_collection,source_is_lossy,source_access_mode,prepared_path,path,width,height,dtype,pixel_sha256
0,1,test,zip:///workspace/data/raw/bossbase.zip!/graysc...,jpg,grayscale/QF95,True,zip-direct,/workspace/data/processed/BOSSbase_primary/png...,/workspace/data/processed/BOSSbase_primary/png...,256,256,uint8,7cdea2c51023efde1a2909f9bdb5b6576f1801192dbb11...
1,2,train,zip:///workspace/data/raw/bossbase.zip!/graysc...,jpg,grayscale/QF95,True,zip-direct,/workspace/data/processed/BOSSbase_primary/png...,/workspace/data/processed/BOSSbase_primary/png...,256,256,uint8,06517138e5fde348971ca8d1bf78d1e350f375b69289e1...
2,3,train,zip:///workspace/data/raw/bossbase.zip!/graysc...,jpg,grayscale/QF95,True,zip-direct,/workspace/data/processed/BOSSbase_primary/png...,/workspace/data/processed/BOSSbase_primary/png...,256,256,uint8,4ba0bc3a4c85ae1ba5d5826b65bf40bf7f4edd9227bcd1...
3,4,train,zip:///workspace/data/raw/bossbase.zip!/graysc...,jpg,grayscale/QF95,True,zip-direct,/workspace/data/processed/BOSSbase_primary/png...,/workspace/data/processed/BOSSbase_primary/png...,256,256,uint8,60b085159d6155e827b0d919c9e0bebe8452ed22511c8a...
4,5,test,zip:///workspace/data/raw/bossbase.zip!/graysc...,jpg,grayscale/QF95,True,zip-direct,/workspace/data/processed/BOSSbase_primary/png...,/workspace/data/processed/BOSSbase_primary/png...,256,256,uint8,5266da113950c9b0fd0eb450f47f5ae3ebf87a89ad61f7...


### Output contract

Subsequent notebooks use the stable primary manifest:

`/workspace/data/processed/dataset_manifest.csv`

Do **not** reshuffle the dataset later. Every derivative of a source image must retain the split assigned here. If the source is JPEG, exact reversibility is defined relative to the decoded 8-bit pixel array loaded from that JPEG; PNG export is then verified losslessly.
